# Lab 1

## Parte 2: Tensores en PyTorch y tal
### 2.1 Tensores en PyTorch

In [2]:
import torch
import numpy as np


In [3]:
# --- Crear tensores ---
t_zeros = torch.zeros(3, 4) # tensor de ceros 3x4
t_rand = torch.rand(2, 3) # valores aleatorios uniformes [0,1)
t_manual = torch.tensor([[1.0, 2.0], [3.0, 4.0]]) # desde lista Python
print("Shape:", t_rand.shape) # torch.Size([2, 3])
print("dtype:", t_rand.dtype) # torch.float32
print("device:", t_rand.device) # cpu


Shape: torch.Size([2, 3])
dtype: torch.float32
device: cpu


In [4]:
# --- Operaciones elementales ---
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([4.0, 5.0, 6.0])
print(a + b) # tensor([5., 7., 9.])
print(a * b) # multiplicación elemento a elemento
print(torch.dot(a, b)) # producto punto: 1*4 + 2*5 + 3*6 = 32

tensor([5., 7., 9.])
tensor([ 4., 10., 18.])
tensor(32.)


In [5]:
# --- Interoperabilidad con NumPy ---
np_arr = np.array([1.0, 2.0, 3.0])
t_from_np = torch.from_numpy(np_arr) # comparten memoria
np_from_t = t_manual.numpy() # de vuelta a numpy

### 2.2 Autograd: diferenciación automática

In [8]:
#requires_grad=True le dice a PyTorch que queremos calcular gradientes
x = torch.tensor(2.0, requires_grad=True)
w = torch.tensor(3.0, requires_grad=True)
b = torch.tensor(1.0, requires_grad=True)
# Función: y = w*x + b (una neurona sin activación)
y = w * x + b
print("y =", y.item()) # 7.0
# Calcular gradientes respecto a y
y.backward()
# dy/dx = w = 3.0
# dy/dw = x = 2.0
# dy/db = 1.0
print("grad x:", x.grad) # tensor(3.) 
print("grad w:", w.grad) # tensor(2.) 
print("grad b:", b.grad) # tensor(1.) 
# IMPORTANTE: limpiar gradientes antes del siguiente paso 
x.grad.zero_() 
w.grad.zero_() 
b.grad.zero_()

y = 7.0
grad x: tensor(3.)
grad w: tensor(2.)
grad b: tensor(1.)


tensor(0.)

## Parte 3: Tensores en TensorFlow
### Operaciones equivalentes en TensorFlow

In [9]:
import tensorflow as tf 
# --- Crear tensores --- 
t_zeros = tf.zeros((3, 4)) 
t_rand = tf.random.uniform((2, 3)) 
t_const = tf.constant([[1.0, 2.0], [3.0, 4.0]]) 
print("Shape:", t_rand.shape) # (2, 3) 
print("dtype:", t_rand.dtype) # float32 
print("device:", t_rand.device) # CPU:0

Shape: (2, 3)
dtype: <dtype: 'float32'>
device: /job:localhost/replica:0/task:0/device:CPU:0


In [13]:
# --- Operaciones --- 
a = tf.constant([1.0, 2.0, 3.0]) 
b = tf.constant([4.0, 5.0, 6.0]) 
print(a + b) # tf.Tensor([5. 7. 9.], ...)
print(tf.tensordot(a, b, axes=1)) # producto punto 
# alternativa: tf.tensordot(a, b, axes=1) 


tf.Tensor([5. 7. 9.], shape=(3,), dtype=float32)
tf.Tensor(32.0, shape=(), dtype=float32)


In [14]:
# --- GradientTape: equivalente al autograd de PyTorch --- 
x = tf.Variable(2.0) # Variable = tensor entrenable 
w = tf.Variable(3.0) 
b_val = tf.Variable(1.0) 

with tf.GradientTape() as tape: 
    y = w * x + b_val 

grads = tape.gradient(y, [x, w, b_val]) 
print("grad x:", grads[0].numpy()) # 3.0 
print("grad w:", grads[1].numpy()) # 2.0 
print("grad b:", grads[2].numpy()) # 1.0

grad x: 3.0
grad w: 2.0
grad b: 1.0


## Parte 4: Red Neuronal Mínima (1 neurona)
### 4.1 Una neurona en PyTorch

In [15]:
import torch
import torch.nn as nn
 
# Una capa lineal: 3 entradas → 1 salida
neurona = nn.Linear(in_features=3, out_features=1)
 
# Ver parámetros inicializados aleatoriamente
print("Pesos:", neurona.weight)   # shape: (1, 3)
print("Bias:",  neurona.bias)     # shape: (1,)
 
# Forward pass con un batch de 5 ejemplos
x = torch.rand(5, 3)   # 5 muestras, 3 features
y_pred = neurona(x)
print("Salida:", y_pred.shape)   # torch.Size([5, 1])
print(y_pred)

Pesos: Parameter containing:
tensor([[-0.1225,  0.0613,  0.0206]], requires_grad=True)
Bias: Parameter containing:
tensor([-0.2466], requires_grad=True)
Salida: torch.Size([5, 1])
tensor([[-0.2608],
        [-0.2492],
        [-0.2650],
        [-0.1850],
        [-0.2667]], grad_fn=<AddmmBackward0>)


### 4.2 Una neurona en Keras

In [16]:
import tensorflow as tf
 
# Una capa densa: 3 entradas → 1 salida
neurona = tf.keras.layers.Dense(units=1, input_shape=(3,))
 
# Forward pass
import numpy as np
x = np.random.rand(5, 3).astype(np.float32)
y_pred = neurona(x)
print("Salida:", y_pred.shape)   # (5, 1)
print(y_pred.numpy())
 
# Ver parámetros
print("Pesos:", neurona.kernel.shape)   # (3, 1)
print("Bias:",  neurona.bias.shape)     # (1,)


c:\Users\Lenovo\anaconda3\envs\frameworks-ia\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Salida: (5, 1)
[[0.46552035]
 [0.9890395 ]
 [0.5807537 ]
 [1.4513291 ]
 [0.5131686 ]]
Pesos: (3, 1)
Bias: (1,)
